# Notebook 08 — Model D: BiLSTM + Label Attention (LAAT)

**Architecture**: Implements the Label Attention Model from [Vu et al., IJCAI 2020](https://www.ijcai.org/proceedings/2020/461).

**Key differences from Model C (BERT-based)**:
- **Encoder**: BiLSTM (trained from scratch) instead of pre-trained ClinicalBERT
- **Input**: Full document as word tokens (up to 4,000 words) — no chunking needed
- **Embeddings**: Word-level (learned) instead of subword BPE
- **Label Attention**: Same concept — per-label query vectors attend over encoder hidden states

**Pipeline**: Word tokens → Embedding → BiLSTM → Label Attention → Per-label FFN → Sigmoid

## 1. Config & GPU Check

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

from src.config import (
    DATA_DIR, MODEL_D_DIR, SEED,
    MODEL_D_EMBED_DIM, MODEL_D_HIDDEN_DIM, MODEL_D_NUM_LAYERS,
    MODEL_D_ATTN_DIM, MODEL_D_DROPOUT, MODEL_D_LR, MODEL_D_EPOCHS,
    MODEL_D_BATCH_SIZE, MODEL_D_GRAD_ACCUM, MODEL_D_MAX_TOKENS,
    MODEL_D_VOCAB_SIZE, MODEL_D_EARLY_STOP,
    MODEL_D_FOCAL_GAMMA, MODEL_D_FOCAL_ALPHA,
    USE_AMP,
)
from src.train import set_seed

SAVE_DIR = MODEL_D_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('No GPU detected — BiLSTM is lighter than BERT but GPU still recommended')

print(f'\nModel D Config:')
print(f'  Embedding dim:   {MODEL_D_EMBED_DIM}')
print(f'  BiLSTM hidden:   {MODEL_D_HIDDEN_DIM} (x2 bidirectional = {MODEL_D_HIDDEN_DIM * 2})')
print(f'  Attention dim:   {MODEL_D_ATTN_DIM}')
print(f'  Max tokens:      {MODEL_D_MAX_TOKENS}')
print(f'  Vocab size:      {MODEL_D_VOCAB_SIZE}')
print(f'  Batch size:      {MODEL_D_BATCH_SIZE}')
print(f'  Epochs:          {MODEL_D_EPOCHS} (early stop patience={MODEL_D_EARLY_STOP})')
print(f'  Learning rate:   {MODEL_D_LR}')
print(f'  Dropout:         {MODEL_D_DROPOUT}')
print(f'  Device:          {device}')

## 2. Load Data & Labels

In [ ]:
from src.data import load_splits, load_label_binarizer, build_label_matrix

train_df, val_df, test_df = load_splits()
mlb = load_label_binarizer()
vocab = list(mlb.classes_)
NUM_LABELS = len(vocab)

Y_train = build_label_matrix(train_df, mlb)
Y_val   = build_label_matrix(val_df, mlb)
Y_test  = build_label_matrix(test_df, mlb)

print(f'Labels: {NUM_LABELS}')
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print(f'Y_train shape: {Y_train.shape}')
print(f'Label density: {Y_train.mean():.4f}')

## 3. Build Word Vocabulary

Unlike Models B/C which use BERT's subword tokenizer, the BiLSTM model uses word-level tokenization. We build a vocabulary from the training set (top 50K most frequent words).

In [ ]:
from src.data import build_word_vocab

# Build vocabulary from training texts only (no data leakage)
word2idx = build_word_vocab(train_df['clean_text'], max_vocab_size=MODEL_D_VOCAB_SIZE)

# Save for reuse (inference, API, etc.)
with open(DATA_DIR / 'word_vocab.pkl', 'wb') as f:
    pickle.dump(word2idx, f)
print(f'Saved word_vocab.pkl ({len(word2idx):,} words)')

# Quick stats
train_lengths = train_df['clean_text'].apply(lambda t: len(t.split()))
print(f'\nDocument length stats (words):')
print(f'  Mean:   {train_lengths.mean():.0f}')
print(f'  Median: {train_lengths.median():.0f}')
print(f'  95th:   {train_lengths.quantile(0.95):.0f}')
print(f'  Max:    {train_lengths.max()}')
print(f'  > {MODEL_D_MAX_TOKENS} tokens: {(train_lengths > MODEL_D_MAX_TOKENS).mean():.1%} of docs (will be truncated)')

## 4. Create Datasets & DataLoaders

In [ ]:
from src.data import BiLSTMDataset
from torch.utils.data import DataLoader

train_ds = BiLSTMDataset(train_df['clean_text'], Y_train, word2idx, max_tokens=MODEL_D_MAX_TOKENS)
val_ds   = BiLSTMDataset(val_df['clean_text'],   Y_val,   word2idx, max_tokens=MODEL_D_MAX_TOKENS)
test_ds  = BiLSTMDataset(test_df['clean_text'],  Y_test,  word2idx, max_tokens=MODEL_D_MAX_TOKENS)

train_loader = DataLoader(train_ds, batch_size=MODEL_D_BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=MODEL_D_BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=MODEL_D_BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

# Verify a single sample
sample = train_ds[0]
print(f'\nSample shapes:')
print(f'  input_ids:      {sample["input_ids"].shape}')       # (max_tokens,)
print(f'  attention_mask: {sample["attention_mask"].shape}')   # (max_tokens,)
print(f'  labels:         {sample["labels"].shape}')           # (num_labels,)
print(f'  real tokens:    {sample["attention_mask"].sum().item()}')

## 5. Build BiLSTM-LAAT Model

In [ ]:
from src.models import BiLSTMLAAT

model = BiLSTMLAAT(
    vocab_size=len(word2idx),
    num_labels=NUM_LABELS,
    embed_dim=MODEL_D_EMBED_DIM,
    hidden_dim=MODEL_D_HIDDEN_DIM,
    num_layers=MODEL_D_NUM_LAYERS,
    attn_dim=MODEL_D_ATTN_DIM,
    dropout=MODEL_D_DROPOUT,
).to(device)

total_params   = sum(p.numel() for p in model.parameters())
trainable      = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable:>12,}')
print(f'\nFor comparison:')
print(f'  Model B (ClinicalBERT):        108,348,722')
print(f'  Model C (Chunk+LabelAttn):     108,387,122')
print(f'  Model D (BiLSTM-LAAT):         {trainable:>12,}  ← much lighter!')

# Smoke test: forward pass
with torch.no_grad():
    test_batch = next(iter(train_loader))
    test_ids = test_batch['input_ids'][:2].to(device)
    test_mask = test_batch['attention_mask'][:2].to(device)
    test_out = model(test_ids, test_mask)
    print(f'\nSmoke test: input {test_ids.shape} → output {test_out.shape}  ✓')

## 6. Training

Single-phase training (no frozen backbone — BiLSTM is trained from scratch). Uses focal loss for class imbalance handling and early stopping to prevent overfitting.

In [ ]:
from src.train import train_model

print('='*60)
print('Training BiLSTM-LAAT with Focal Loss + Early Stopping')
print('='*60)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=SAVE_DIR,
    lr=MODEL_D_LR,
    epochs=MODEL_D_EPOCHS,
    grad_accum=MODEL_D_GRAD_ACCUM,
    warmup_ratio=0.05,
    use_focal_loss=True,
    focal_gamma=MODEL_D_FOCAL_GAMMA,
    focal_alpha=MODEL_D_FOCAL_ALPHA,
    checkpoint_every=5000,
    is_chunked=False,           # BiLSTM is NOT chunked
    early_stopping_patience=MODEL_D_EARLY_STOP,
    device=device,
)

## 7. Training History

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(hist_df['epoch'], hist_df['train_loss'], 'b-o', markersize=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train Loss'); ax1.set_title('Training Loss')

# Val F1 (tuned vs default)
ax2.plot(hist_df['epoch'], hist_df['val_f1_tuned'], 'g-o', markersize=4, label='Tuned threshold')
ax2.plot(hist_df['epoch'], hist_df['val_f1_at_0.5'], 'gray', linestyle='--', alpha=0.5, label='F1 @ t=0.5')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Micro-F1'); ax2.set_title('Validation F1')
ax2.legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_curves.png', dpi=150)
plt.show()
print(hist_df.to_string(index=False))

## 8. Temperature Scaling (Calibration)

In [ ]:
from src.models import TemperatureScaler
from src.train import collect_logits
from src.evaluate import tune_global_threshold, expected_calibration_error
import torch as th

# Reload best model
model.load_state_dict(torch.load(SAVE_DIR / 'best_model.pt', map_location=device))
model.eval()

# Collect raw logits on validation set
print('Collecting validation logits...')
val_logits, val_labels = collect_logits(
    model, val_loader, device=device, use_amp=USE_AMP, is_chunked=False,
)

# Fit temperature scaler
scaler = TemperatureScaler()
T = scaler.fit(val_logits, val_labels)

# Compare calibration before and after
probs_uncal = th.sigmoid(th.tensor(val_logits)).numpy()
probs_cal   = scaler.calibrate(val_logits)

ece_before = expected_calibration_error(probs_uncal, val_labels)
ece_after  = expected_calibration_error(probs_cal, val_labels)

t_before, f1_before = tune_global_threshold(probs_uncal, val_labels)
t_after, f1_after   = tune_global_threshold(probs_cal, val_labels)

print(f'\nCalibration Results:')
print(f'  Before: ECE={ece_before:.4f}  best_t={t_before:.3f}  F1={f1_before:.4f}')
print(f'  After:  ECE={ece_after:.4f}  best_t={t_after:.3f}  F1={f1_after:.4f}')
print(f'  Temperature: {T:.4f}')

# Save temperature
import json
with open(SAVE_DIR / 'temperature.json', 'w') as f:
    json.dump({'temperature': T, 'ece_before': ece_before, 'ece_after': ece_after}, f, indent=2)

## 9. Test Evaluation

In [ ]:
from src.evaluate import full_metrics, tune_per_label_threshold
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

# Collect test logits
print('Computing test predictions...')
test_logits, Y_test_np = collect_logits(
    model, test_loader, device=device, use_amp=USE_AMP, is_chunked=False,
)

# Calibrated probabilities
P_test_cal = scaler.calibrate(test_logits)
# Uncalibrated probabilities
P_test_uncal = th.sigmoid(th.tensor(test_logits)).numpy()

# Global threshold (on calibrated probs, tuned on val)
best_t_global, _ = tune_global_threshold(probs_cal, val_labels)
results_global = full_metrics(P_test_cal, Y_test_np, best_t_global, 'Model D (BiLSTM-LAAT)')

print('\n=== Test Results (Global Threshold) ===')
for k, v in results_global.items():
    print(f'  {k:15s}: {v}')

# Save results
with open(SAVE_DIR / 'test_results.json', 'w') as f:
    json.dump(results_global, f, indent=2)
np.save(SAVE_DIR / 'P_test_calibrated.npy', P_test_cal)
np.save(SAVE_DIR / 'P_test_uncalibrated.npy', P_test_uncal)
np.save(SAVE_DIR / 'P_val_best.npy', probs_cal)
np.save(SAVE_DIR / 'Y_val_best.npy', val_labels)
print('\nResults saved to', SAVE_DIR)

## 10. Head / Torso / Tail Analysis

In [ ]:
preds_test = (P_test_cal >= best_t_global).astype(int)
per_label_f1 = f1_score(Y_test_np, preds_test, average=None, zero_division=0)
per_label_freq = Y_train.sum(0)

label_df = pd.DataFrame({
    'icd_code': vocab, 'train_freq': per_label_freq, 'test_f1': per_label_f1,
})

print(f'{"Bucket":25s}  {"n":>6}  {"Avg F1":>7}')
print('-' * 45)
for lo, hi, name in [(500, 1e9, 'head (>=500)'), (100, 499, 'torso (100-499)'), (0, 99, 'tail (<100)')]:
    s = label_df[(label_df['train_freq'] >= lo) & (label_df['train_freq'] <= hi)]
    if len(s) == 0:
        continue
    print(f'{name:25s}  {len(s):6d}  {s["test_f1"].mean():7.4f}')

plt.figure(figsize=(7, 4))
plt.scatter(label_df['train_freq'].clip(upper=2000), label_df['test_f1'], alpha=0.3, s=8)
plt.xscale('log'); plt.xlabel('Train freq (log)'); plt.ylabel('Test F1')
plt.title('Per-label F1 vs training frequency (Model D: BiLSTM-LAAT)')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'head_tail_f1.png', dpi=120)
plt.show()

## 11. Compare All Models

Side-by-side comparison: Model A (TF-IDF), Model B (ClinicalBERT), Model C (Chunk+BERT+LabelAttn), Model D (BiLSTM+LabelAttn).

In [ ]:
from src.config import MODEL_A_DIR, MODEL_B_DIR, MODEL_C_DIR

# Load other models' results
with open(MODEL_A_DIR / 'results.json') as f:
    res_a = json.load(f)
with open(MODEL_B_DIR / 'test_results.json') as f:
    res_b = json.load(f)

# Model C v1
res_c1 = None
try:
    with open(MODEL_C_DIR / 'test_results.json') as f:
        res_c1 = json.load(f)
except FileNotFoundError:
    pass

# Model C v2
res_c2 = None
try:
    with open(MODEL_C_DIR / 'v2' / 'test_results.json') as f:
        c2_data = json.load(f)
        res_c2 = c2_data.get('global_threshold', c2_data)
except FileNotFoundError:
    pass

rows = [
    {'Model': 'A: TF-IDF + SGD',
     'Micro-F1': res_a['test']['micro_f1'],
     'Macro-F1': res_a['test']['macro_f1'],
     'AUROC': res_a['test']['micro_auroc']},
    {'Model': 'B: ClinicalBERT',
     'Micro-F1': res_b['micro_f1'],
     'Macro-F1': res_b['macro_f1'],
     'AUROC': res_b['micro_auroc']},
]

if res_c1:
    rows.append({
        'Model': 'C v1: Chunk+BERT+Attn',
        'Micro-F1': res_c1.get('Micro-F1', 0),
        'Macro-F1': res_c1.get('Macro-F1', 0),
        'AUROC': res_c1.get('Micro-AUROC', 0),
    })

if res_c2:
    rows.append({
        'Model': 'C v2: Fixed+Focal',
        'Micro-F1': res_c2.get('Micro-F1', 0),
        'Macro-F1': res_c2.get('Macro-F1', 0),
        'AUROC': res_c2.get('Micro-AUROC', 0),
    })

rows.append({
    'Model': 'D: BiLSTM-LAAT',
    'Micro-F1': results_global['Micro-F1'],
    'Macro-F1': results_global['Macro-F1'],
    'AUROC': results_global['Micro-AUROC'],
})

comparison = pd.DataFrame(rows)
print('\n=== All Models Comparison (Test Set) ===')
print(comparison.to_string(index=False))
comparison.to_csv(SAVE_DIR / 'comparison_all.csv', index=False)